# 04 - Donut inference demo

Demo inference Donut tren anh bien lai trong MC-OCR test split.

In [ ]:
import json
import os
import sys
import torch
from PIL import Image
import matplotlib.pyplot as plt
from transformers import DonutProcessor, VisionEncoderDecoderModel

sys.path.insert(0, '..')
from scripts.utils import parse_donut_output

CHECKPOINT = 'results/e2_donut/checkpoints/mcocr'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

processor = DonutProcessor.from_pretrained(CHECKPOINT)
model = VisionEncoderDecoderModel.from_pretrained(CHECKPOINT).to(device)
model.eval()

In [ ]:
def predict(image_path):
    image = Image.open(image_path).convert('RGB')
    pixel_values = processor(image, return_tensors='pt').pixel_values.to(device)
    prompt_ids = processor.tokenizer('<s_mcocr>', add_special_tokens=False, return_tensors='pt').input_ids.to(device)
    with torch.no_grad():
        gen = model.generate(
            pixel_values,
            decoder_input_ids=prompt_ids.repeat(pixel_values.shape[0], 1),
            max_length=model.config.decoder.max_position_embeddings,
            pad_token_id=processor.tokenizer.pad_token_id,
            eos_token_id=processor.tokenizer.eos_token_id,
        )
    text = processor.tokenizer.decode(gen[0], skip_special_tokens=False)
    return parse_donut_output(text, '<s_mcocr>')

In [ ]:
test_dir = 'data/mc-ocr/donut_format/test'
metadata_path = os.path.join(test_dir, 'metadata.jsonl')

if os.path.exists(metadata_path):
    with open(metadata_path, 'r', encoding='utf-8') as f:
        records = [json.loads(line) for line in f if line.strip()][:5]

    fig, axes = plt.subplots(1, len(records), figsize=(4 * len(records), 6))
    if len(records) == 1:
        axes = [axes]
    for ax, rec in zip(axes, records):
        img_path = os.path.join(test_dir, rec['file_name'])
        if os.path.exists(img_path):
            ax.imshow(Image.open(img_path))
            pred = predict(img_path)
            info = '\n'.join(f'{k}: {v}' for k, v in pred.items() if v)
            ax.set_xlabel(info, fontsize=7)
        ax.set_title(rec['file_name'], fontsize=7)
        ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout(); plt.show()
else:
    print('No test data found')